In [1]:
import pandas as pd 
import numpy as np 
import os
import pickle


In [ ]:
# 导入自定义模块中的函数
from GenSimKWFreq import gen_similar_word_freq_csv,open_csv_df,gen_eng_compressed_csv,gen_llm_word_def_csv
from BinVectorMarker import BinVectorMarker

In [ ]:
# 保存任意 Python 对象为 pickle 文件
def save_to_pickle(obj_name:str, data) -> None:
    """
        Descr:
            This function will just save a 
        Args:
        Ret:
    将对象保存为 pickle 文件
    参数:
        obj_name: 保存的文件名（不含扩展名）
        data: 要保存的数据对象
    返回:
        None
    """
    out_dir = 'CACHED_PY_OBJS'
    os.makedirs(out_dir,exist_ok=True)

    fp = os.path.join(os.getcwd(),out_dir,f"{obj_name}.pkl")
    with open(fp, "wb") as f:
        pickle.dump(data, f)

# 从 pickle 文件中加载对象
def get_pkl_obj(obj_name ) :
    out_dir = 'CACHED_PY_OBJS'

    fp = os.path.join(os.getcwd(),out_dir,f"{obj_name}.pkl")
    with open(fp, "rb") as f:
        data = pickle.load(f)

    return data

In [ ]:
# 生成关键字频率和定义相关的 CSV 文件
def gen_csvs(sample_descs:pd.Series,out_dir='LITHO_CSVS',use_cache_if_exists=True) :
    
    CWD:str            = os.getcwd()
    OUT_CSV_FOLDER:str = os.path.join(CWD,out_dir)

    DEFAULT_COL:str  = '0'
    RED_COL:str      = '31'

    SIMILAR_KEY_WORD_FREQ:str           = "similar_keywords_freq.csv"
    FILITERED_SIMILAR_KEY_WORD_FREQ:str = "similar_keywords_compressed_freq.csv"

    SIMILAR_KEY_WORD_FREQ_FP:str    = os.path.join(OUT_CSV_FOLDER,SIMILAR_KEY_WORD_FREQ) 
    FILITERED_SIMILAR_KEY_WORD_FREQ_FP:str = os.path.join(OUT_CSV_FOLDER,FILITERED_SIMILAR_KEY_WORD_FREQ)

    CACHED_PY_OBJS = "CACHED_PY_OBJS"

    # Create the csv folders
    os.makedirs(OUT_CSV_FOLDER,exist_ok=True)

    # Create the cached py obj folder
    os.makedirs(CACHED_PY_OBJS,exist_ok=True)


    # Create the similar_keywords_freq.csv file
    gen_similar_word_freq_csv(sample_descs,SIMILAR_KEY_WORD_FREQ_FP,use_cache_if_exists)


    # Create the filtered "similar_keywords_freq.csv" csv file which is called "similar_keywords_compressed_freq.csv"
    gen_eng_compressed_csv(SIMILAR_KEY_WORD_FREQ_FP,FILITERED_SIMILAR_KEY_WORD_FREQ_FP)
    
    # Query the open AI API to get 'word_def.csv'
    req_fp = os.path.join(OUT_CSV_FOLDER,'gpt_batch_req.jsonl')
    res_fp = os.path.join(OUT_CSV_FOLDER,'gpt_batch_res.jsonl')
    out_fp = os.path.join(OUT_CSV_FOLDER,'word_def.csv')

    gen_llm_word_def_csv(FILITERED_SIMILAR_KEY_WORD_FREQ_FP,req_fp,res_fp,out_fp)

In [ ]:
CWD             = os.getcwd()
BRIT_DF_PATH    = os.path.join(CWD,"lithogeochem_data.csv")

brit_df       = open_csv_df(BRIT_DF_PATH)

df:pd.Series = pd.concat([
                brit_df['Sample_Desc'], 
                # usgs_df['ADDL_ATTR'],
                # usgs_df['SPEC_NAME'],
                # usgs_df["XNDRYCLASS"],
                # sarig_df['feature'],
                # ontario_df['ROCK'],
                # ontario_df['ROCK.1'],
                # ontario_df['ROCK.2']
                ],axis=0)

print(df.shape)

csv_out_folder = 'LITHO_CSVS'
if not(os.path.exists(csv_out_folder)):
    gen_csvs(df,out_dir=csv_out_folder)



(10914,)
Generating Defn word graph for BinVectorMarker


100%|██████████| 733/733 [00:00<00:00, 12563.03it/s]


Generating 'similar_key_word_map' and 'def_word_idx_map'


  0%|          | 1/10914 [00:00<16:15, 11.19it/s]


LookupError: 
**********************************************************************
  Resource [93muniversal_tagset[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('universal_tagset')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtaggers/universal_tagset/en-ptb.map[0m

  Searched in:
    - '/home/tongweizj/nltk_data'
    - '/home/tongweizj/miniconda3/envs/mineral-PyTorch/nltk_data'
    - '/home/tongweizj/miniconda3/envs/mineral-PyTorch/share/nltk_data'
    - '/home/tongweizj/miniconda3/envs/mineral-PyTorch/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
    - ''
**********************************************************************


In [ ]:
sim_key_word_csv_path = os.path.join(CWD,csv_out_folder,"similar_keywords_compressed_freq.csv")
word_def_csv_path     = os.path.join(CWD,csv_out_folder,"word_def.csv")

BIN_VEC_MARKER = BinVectorMarker(sim_key_word_csv_path,word_def_csv_path)


In [ ]:
bin_vecs= BIN_VEC_MARKER.gen_bin_vecs(df).apply(lambda x: ','.join(map(str, x)))
bin_vec_df = pd.concat([df,bin_vecs],keys=['Sample_Descr','Bin_Vec'],axis=1)
bin_vec_df.to_csv("bin_vec.csv")